### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [9]:
import os

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

### Ryn with thread id
config={"configurable": {"thread_id": "test-1"}}


In [11]:
questions = [
    "What is 2+2 ?",
    "What is 10*5 ?",
    "What is 100/5 ?",
    "What is 15-7 ?",
    "What is 3*3 ?",
    "What is 4*4 ?"
]

for q in questions:
    response=agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2 ?', additional_kwargs={}, response_metadata={}, id='2a33360c-6e4a-4807-b9e9-2f0d0cd1aa4f'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2 ?"\n2.  **Identify Core Task:** Simple arithmetic addition.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Nuance/Context:** No special context, just a straightforward math question.\n6.  **Output Generation:** "2 + 2 equals 4." (or simply "4")\n\nFinal decision: Keep it direct and accurate. "2 + 2 = 4" or "The answer is 4." matches the simplicity of the question.✅\n</think>\n\n2 + 2 = 4', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 17, 'total_tokens': 185, 'completion_time': 0.325324918, 'completion_tokens_details': None, 'prompt_time': 0.000909873, 'prompt_tokens_details': None, 'queue_tim

### Token size

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [13]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~120 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='98572c02-6561-447f-9605-40b527e67844'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to find hotels in Paris.\n2.  **Identify Available Tools**: `search_hotels` function is available.\n3.  **Check Tool Parameters**: `search_hotels` requires a `city` parameter (string).\n4.  **Extract Parameters**: `city` = "Paris".\n5.  **Call Tool**: `search_hotels(city="Paris")`.\n6.  **Generate Response**: Wait for tool output, then format and present to user. (Self-correction/Refinement: I will just call the tool now).✅\n', 'tool_calls': [{'id': 'vkmebyksa', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 277, 'total_tokens': 440, 'completion_time': 0.310218991, 'completio

KeyboardInterrupt: 

### Fraction

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

ValueError: Model profile information is required to use fractional token limits, and is unavailable for the specified model. Please use absolute token counts instead, or pass `

ChatModel(..., profile={"max_input_tokens": ...})`.

with a desired integer value of the model's maximum input tokens.

### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='77de20de-8568-4970-8209-08f1fe0efcf5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: john@test.com\n   - Subject: \'Hello\'\n   - Body: \'How are you?\'\n2.  **Identify Available Tools**: `send_email_tool` matches the requirement.\n   - Parameters: `recipient` (string), `subject` (string), `body` (string). All required.\n3.  **Map Input to Tool**:\n   - `recipient`: "john@test.com"\n   - `subject`: "Hello"\n   - `body`: "How are you?"\n4.  **Execute Tool Call**: Call `send_email_tool` with the mapped parameters.\n5.  **Formulate Response**: Return the tool call. No extra text needed.✅\n', 'tool_calls': [{'id': '51erdxx55', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}'

### Resolve

In [ ]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to john@test.com with the subject 'Hello' and body 'How are you?'.


### Reject

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Rejecting...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Approving...
✅ Result: It seems you rejected the email sending action. How can I help you further?


### Editing

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='72d77526-47fe-45db-b230-c168adb6d493'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI need to call `send_email_tool` with recipient="john@test.com", subject="Hello", and body="How are you?".\nAll required parameters are provided.\nI will construct the tool call now.✅\n', 'tool_calls': [{'id': '4g80gyhrw', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 374, 'total_tokens': 487, 'completion_time': 0.214956473, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.029084676, 'prompt_tokens_details': None, 'queue_time': 0.046866693, 'total_time': 0.244041149}, 'model_name': 'qwen/qwen

In [32]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Editting...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "coorect@gmail.com",
                                "subject": "Corrected subject",
                                "body": "this email has been editted."
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

    print(result)
    
    print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Editting...
{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='72d77526-47fe-45db-b230-c168adb6d493'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email.\nI need to call `send_email_tool` with recipient="john@test.com", subject="Hello", and body="How are you?".\nAll required parameters are provided.\nI will construct the tool call now.✅\n', 'tool_calls': [{'id': '4g80gyhrw', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 374, 'total_tokens': 487, 'completion_time': 0.214956473, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.029084676, 'prompt_tokens_details': None, 'queue_time': 0.046866693, 'total_time': 0.244041149}, 'mo